In [2]:
!pip install kagglehub pandas scikit-learn

import kagglehub
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import difflib

In [4]:
path = kagglehub.dataset_download("harshshinde8/movies-csv")
df = pd.read_csv(f"{path}/movies.csv")

selected_features = ["genres", "keywords", "tagline", "cast", "director"]
for feature in selected_features:
    df[feature] = df[feature].fillna("")

combined_features = (
    df["genres"]
    + " "
    + df["keywords"]
    + " "
    + df["tagline"]
    + " "
    + df["cast"]
    + " "
    + df["director"]
)

vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)

model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(feature_vectors)


def recommend_movies(movie_name, num_recommendations=5):
    movie_list = df["title"].tolist()
    find_close_match = difflib.get_close_matches(movie_name, movie_list)

    if not find_close_match:
        print("Movie not found in dataset.")
        return

    close_match = find_close_match[0]
    index_of_movie = df[df.title == close_match]["index"].values[0]

    movie_vector = feature_vectors[index_of_movie]
    distances, indices = model.kneighbors(
        movie_vector, n_neighbors=num_recommendations + 1
    )

    print(f"Recommendations for '{close_match}':\n")
    i = 1
    for idx in indices.flatten():
        title = df.iloc[idx]["title"]
        if title != close_match:
            print(f"{i}. {title}")
            i += 1


recommend_movies("Alien")

Using Colab cache for faster access to the 'movies-csv' dataset.
Recommendations for 'Alien':

1. Moonraker
2. Aliens
3. Alien³
4. Planet of the Apes
5. Avatar
